## Förbättra mixandMatch description
* [#33](https://github.com/salgo60/ifkdb/issues/33)
* Denna notebook [33_difhistoryplayer](https://github.com/salgo60/ifkdb/blob/main/Notebook/33_difhistoryplayer.ipynb)

In [1]:
from datetime import datetime
import platform
import sys
import os

# Start timer
start_time = datetime.now()
print(f"Start time: {start_time:%Y-%m-%d %H:%M:%S}")


Start time: 2026-03-05 23:58:08


In [2]:
filename="difplayers.json"

In [3]:
import json
from pprint import pprint

# Read JSON
with open(filename, "r", encoding="utf-8") as f:
    data = json.load(f)

# Show top-level structure
print("Top level type:", type(data))

# Count items
if isinstance(data, list):
    print("Number of items:", len(data))
elif isinstance(data, dict):
    print("Number of keys:", len(data))
else:
    print("Unknown JSON structure")

# Show structure sample
print("\nSample structure:")
if isinstance(data, list) and len(data) > 0:
    pprint(data[0])
elif isinstance(data, dict):
    pprint(list(data.keys()))

Top level type: <class 'list'>
Number of items: 1105

Sample structure:
{'auxiliary_data': None,
 'born': None,
 'catalog': '7278',
 'dates_in_wikidata': None,
 'description': '',
 'died': None,
 'entry_id': '225596324',
 'entry_type': 'Q5',
 'external_id': 'marja-aaltonen',
 'external_url': 'https://difhistoria.se/marja-aaltonen',
 'lat': None,
 'lon': None,
 'matched_by_username': 'Sabelöga',
 'matched_on': '20251217144436',
 'mnm_user_id': '6575119',
 'multi_match_candidates': None,
 'name': 'Marja Aaltonen',
 'q': 'Q24827502'}


In [4]:
import json
import pandas as pd

# Load JSON
with open(filename, "r", encoding="utf-8") as f:
    data = json.load(f)

# Create dataframe
df = pd.DataFrame(data)

# Count values in description field
desc_counts = df["description"].value_counts(dropna=False)

print(desc_counts)

description
    1105
Name: count, dtype: int64


In [5]:
import json
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time

df = pd.DataFrame(data)

headers = {
    "User-Agent": "Mozilla/5.0"
}

def scrape_infobox(url):
    try:
        r = requests.get(url, headers=headers, timeout=20)
        soup = BeautifulSoup(r.text, "html.parser")

        section = soup.find("section", class_="wp-block-uagb-section")

        if not section:
            return None

        text = section.get_text(separator=" ", strip=True)
        return text

    except Exception as e:
        print("Error:", url, e)
        return None


# Loop through rows
results = []

for url in df["external_url"]:
    print("Scraping:", url)
    info = scrape_infobox(url)
    print ("\t",info)
    results.append(info)

    #time.sleep(1)  # be polite

df["infobox"] = results

Scraping: https://difhistoria.se/marja-aaltonen
	 Marja Aaltonen Född: 13 maj 1964 Fotbollsspelare Position: målvakt Nummer: 1 Moderklubb: FC Honka (Finland) 1989 1991
Scraping: https://difhistoria.se/alexander-abrahamsson
	 Nils Oscar Alexander Abrahamsson Född: 7 augusti 1999 Fotbollsspelare Position: back Nummer: 28 Moderklubb: Sunnesta IF 5 säsonger. 2015 2016 2017 2018 2019
Scraping: https://difhistoria.se/albion-ademi
	 Albion Ademi Född: 19 februari 1999 Fotbollsspelare Position: mittfältare Längd: 173 cm Nummer: 11 Moderklubb: TPS (Finland) 2021 2022 2023
Scraping: https://difhistoria.se/gale-agbossoumonde
	 Gale Agbossoumonde Född: 17 november 1991 Fotbollsspelare Position: försvarare Moderklubb: Miami F.C. (USA) 1 säsong. 14 matcher. 0 mål. 2011 8 0
Scraping: https://difhistoria.se/bengt-agfemalm
	 Bengt Frans Agfemalm Född: 24 augusti 1930 Curlingspelare Moderklubb: ? x säsonger. 19xx 19xx
Scraping: https://difhistoria.se/nicolaj-agger
	 Nicolaj Moesgaard Agger Född: 23 okto

In [17]:
# max 255
df["description_new"] = df["infobox"].str.slice(0,250)  
df["infobox_clean"] = (
    df["infobox"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()  
    )
df["description_new2"] = df["infobox_clean"].str.slice(0,250)

In [14]:
df.head()

,entry_id,catalog,external_id,external_url,name,description,entry_type,mnm_user_id,q,matched_on,matched_by_username,auxiliary_data,born,died,dates_in_wikidata,lat,lon,multi_match_candidates,infobox,description_new
0,225596324,7278,marja-aaltonen,https://difhistoria.se/marja-aaltonen,Marja Aaltonen,,Q5,6575119,Q24827502,20251217144436,Sabelöga,None,None,None,None,None,None,None,Marja Aaltonen Född: 13 maj 1964 Fotbollsspela...,Marja Aaltonen Född: 13 maj 1964 Fotbollsspela...
1,225596325,7278,alexander-abrahamsson,https://difhistoria.se/alexander-abrahamsson,Alexander Abrahamsson,,Q5,6575119,Q106475418,20251217144438,Sabelöga,None,None,None,None,None,None,None,Nils Oscar Alexander Abrahamsson Född: 7 augus...,Nils Oscar Alexander Abrahamsson Född: 7 augus...
2,225596326,7278,albion-ademi,https://difhistoria.se/albion-ademi,Albion Ademi,,Q5,6575119,Q26869094,20251217144445,Sabelöga,None,None,None,None,None,None,None,Albion Ademi Född: 19 februari 1999 Fotbollssp...,Albion Ademi Född: 19 februari 1999 Fotbollssp...
3,225596327,7278,gale-agbossoumonde,https://difhistoria.se/gale-agbossoumonde,Gale Agbossoumonde,,Q5,6575119,Q1491788,20251217144537,Sabelöga,None,None,None,None,None,None,None,Gale Agbossoumonde Född: 17 november 1991 Fotb...,Gale Agbossoumonde Född: 17 november 1991 Fotb...
4,225596328,7278,bengt-agfemalm,https://difhistoria.se/bengt-agfemalm,Bengt Agfemalm,,Q5,26041,Q138545451,20260303082020,Jssfrk,None,None,None,None,None,None,None,Bengt Frans Agfemalm Född: 24 augusti 1930 Cur...,Bengt Frans Agfemalm Född: 24 augusti 1930 Cur...


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1105 entries, 0 to 1104
Data columns (total 22 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   entry_id                1105 non-null   object
 1   catalog                 1105 non-null   object
 2   external_id             1105 non-null   object
 3   external_url            1105 non-null   object
 4   name                    1105 non-null   object
 5   description             1105 non-null   object
 6   entry_type              1105 non-null   object
 7   mnm_user_id             712 non-null    object
 8   q                       712 non-null    object
 9   matched_on              712 non-null    object
 10  matched_by_username     712 non-null    object
 11  auxiliary_data          0 non-null      object
 12  born                    0 non-null      object
 13  died                    0 non-null      object
 14  dates_in_wikidata       0 non-null      object
 15  lat 

In [20]:
df["description_new2"]

0       Marja Aaltonen Född: 13 maj 1964 Fotbollsspela...
1       Nils Oscar Alexander Abrahamsson Född: 7 augus...
2       Albion Ademi Född: 19 februari 1999 Fotbollssp...
3       Gale Agbossoumonde Född: 17 november 1991 Fotb...
4       Bengt Frans Agfemalm Född: 24 augusti 1930 Cur...
                              ...                        
1100    Hans Östergren Född: x månad 19xx. Curlingspel...
1101    Lars Östergren Född: x månad 19xx Curlingspela...
1102    Irja Monika Österholm Född: 26 november 1959 F...
1103    Noah Andreas Östlund Född: 11 mars 2004 Ishock...
1104    Bo Thomas Östlund Född: 9 september 1965 Ishoc...
Name: description_new2, Length: 1105, dtype: object

In [25]:
out = df[["external_id","name","description_new2"]].copy()

out = out.rename(columns={
    "external_id":"id",
    "description_new2":"desc"
})

out["desc"] = out["desc"].str.slice(0,150)

out = out[out["desc"].notna()]
csv_outfile="mnm_diffhistory_update_20220306.csv"
out.to_csv(csv_outfile, index=False)

print(csv_outfile)

mnm_diffhistory_update_20220306.csv


In [26]:
out.head()

,id,name,desc
0,marja-aaltonen,Marja Aaltonen,Marja Aaltonen Född: 13 maj 1964 Fotbollsspela...
1,alexander-abrahamsson,Alexander Abrahamsson,Nils Oscar Alexander Abrahamsson Född: 7 augus...
2,albion-ademi,Albion Ademi,Albion Ademi Född: 19 februari 1999 Fotbollssp...
3,gale-agbossoumonde,Gale Agbossoumonde,Gale Agbossoumonde Född: 17 november 1991 Fotb...
4,bengt-agfemalm,Bengt Agfemalm,Bengt Frans Agfemalm Född: 24 augusti 1930 Cur...


In [23]:
# End timer
end_time = datetime.now()
duration = end_time - start_time

print("\n--- Execution Summary ---")
print(f"End time:      {end_time:%Y-%m-%d %H:%M:%S}")
print(f"Duration:      {duration}")
print(f"Total seconds: {duration.total_seconds():.2f}")
print(f"Python ver:    {sys.version.split()[0]}")
print(f"Platform:      {platform.system()} {platform.release()}")
print(f"Process ID:    {os.getpid()}")


--- Execution Summary ---
End time:      2026-03-06 11:05:34
Duration:      11:07:25.661561
Total seconds: 40045.66
Python ver:    3.12.2
Platform:      Darwin 25.3.0
Process ID:    96257
